[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/02_langgraph.ipynb)

# Part 2 — Graphs (LangGraph)

In Part 1 the model bounced back and forth between tools until reaching an answer. In many scientific workflows, however, we must progress through a rigid sequence of logic. For example, in finite elements, we go through a `CAD -> mesh -> solve -> postprocess` pipeline. Before progressing from `CAD` to `mesh`, tools must be applied to ensure that a geometry is valid and airtight. Before going from `mesh` to `solve`, diagnostics on mesh quality are evaluated and the meshing is repeated until a valid mesh is obtained.

In this section, we will consider the encoding of these types of workflows as a computational graph that can verify that a sequence of actions is guaranteed before progressing to the end. We highlight that asking a LLM to "not make mistakes" doesn't mean that it won't. The trick to this section is to build guarantees **into the code** rather than ask the LLM to do the right thing and hope for the best.

A **graph** is a directed graph in the standard sense: a set of nodes, plus directed edges specifying which node may follow which. LangGraph executes one by starting at a designated entry node, running the Python function attached to it, following an outgoing edge to the next node, and repeating until it reaches a terminal node. The `CAD -> mesh -> solve -> postprocess` pipeline above is exactly this, with an extra edge running from the mesh diagnostic back to the mesher.

Building one requires three pieces:

- **state** — a `TypedDict` holding every value that has to survive from one node to the next. Here that is the model's answer, whether it passed the check, and two counters.
- **nodes** — Python functions that take the state and return only the keys they changed; LangGraph merges that update in. A node may call the model, or may be plain Python.
- **edges** — the permitted transitions. A plain edge always leads to the same next node. A **conditional edge** instead calls a **router**: a Python function that reads the state and returns the name of the node to run next.

We still call the LLM inside a node, but it no longer has any say in what runs after it. The edges and the router decide that.

**What does this notebook do?** We rebuild the Part 1 arithmetic task as a graph with three nodes — compute, verify, repair — and a router that loops back on failure and gives up after a fixed number of tries. Then we break the compute node on purpose, to watch the guarantee hold.

**You are done when** the broken version stops after a bounded number of tries and reports `verified=False`, instead of looping forever or inventing an answer.

## 2.0 API Setup

Same setup as Part 0, plus `langgraph`. This one also defines `llm_text`, a small helper that sends a prompt and returns the reply as text.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, re
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0" langgraph

from google import genai
from google.genai import types as gtypes

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""

print(f"Gemini client ready (model={MODEL}).")

## 2.1 The syntax

A LangGraph program is assembled in a few moves: declare the state type, register the nodes, say where to start, and wire the edges. `add_conditional_edges` is the interesting one — it takes a router function and a mapping from the router's return value to the next node.

A node receives the whole state and returns only the keys it changed; LangGraph merges that update in. `END` is a sentinel meaning "stop here".

The router is ordinary Python. It reads the state and returns a name. Because it is a normal function, you can read it, test it, and review it in a diff.

## 2.2 The task, as a graph

The task is the arithmetic from Part 1: compute 13 × 47 + 8, which is 619. It is deliberately small so that the graph is the only thing you have to read.

The state carries four values:

| key | meaning |
|---|---|
| `value` | the number the model last returned |
| `verified` | whether that number passed the check |
| `attempts` | how many repairs have run |
| `solves` | how many times the model has been asked |

Three nodes act on that state:

- `n_compute` sends the question to the model, parses a number out of the reply, and writes it to `value`.
- `n_verify` computes 13 × 47 + 8 in Python and compares. It writes `True` or `False` to `verified`, and never looks at how `value` was produced.
- `n_repair` adds one to `attempts`. Its only job is to make the retries countable.

The wiring:

```
compute ──▶ verify ──▶ route ──┬──▶ repair ──▶ compute
                               └──▶ END
```

`route` is the only branch in the graph. It returns `END` when `verified` is true, `END` again when `attempts` has reached `MAX_ATTEMPTS`, and otherwise the string `"repair"`.

Two properties follow from that wiring rather than from anything written in a prompt. First, `verify` runs after every `compute`, because the edge between them is unconditional. Second, the run cannot exceed `MAX_ATTEMPTS` repairs, because `route` is a Python function that counts.

The verifier here is trivial: it recomputes the answer directly, so for this example you never needed the model at all. That keeps the check obvious while you read the graph. The same compute/verify/repair shape appears in `06_agent_hackathon.ipynb` with a verifier that does real work — checking a finite element solution against a manufactured solution.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class GraphState(TypedDict):
    value: float
    attempts: int
    solves: int
    verified: bool

def n_compute(s):
    """The 'agent' node: ask the model and parse a number out of the reply."""
    txt = llm_text("Reply with only a number, no words, no units.",
                   f"What is 13 * 47 + 8? Reply with only the number.")
    m = re.search(r"-?\d+(?:\.\d+)?", txt)
    val = float(m.group()) if m else float('nan')
    return {"value": val, "solves": s["solves"] + 1}

def n_verify(s):
    """Independent check. Recomputes rather than trusting the claim."""
    return {"verified": bool(abs(s["value"] - (13 * 47 + 8)) < 1e-9)}

def n_repair(s):
    return {"attempts": s["attempts"] + 1}

MAX_ATTEMPTS = 3
def route(s):
    if s["verified"]:                 return END
    if s["attempts"] >= MAX_ATTEMPTS: return END      # bounded in code, not in the prompt
    return "repair"

def build_graph(compute_fn):
    g = StateGraph(GraphState)
    g.add_node("compute", compute_fn)
    g.add_node("verify", n_verify)
    g.add_node("repair", n_repair)
    g.set_entry_point("compute")
    g.add_edge("compute", "verify")
    g.add_conditional_edges("verify", route, {"repair": "repair", END: END})
    g.add_edge("repair", "compute")
    return g.compile()

def run_graph(app, state):
    """Stream node-by-node so we can print the path the graph actually took."""
    path = []
    for step in app.stream(state, stream_mode="updates"):
        for node, update in step.items():
            path.append(node)
            state = {**state, **update}
    return state, path

app = build_graph(n_compute)
final, path = run_graph(app, {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(path))
print(f"value={final['value']}  verified={final['verified']}  "
      f"attempts={final['attempts']}  solves={final['solves']}")

## 2.3 Demonstrating a guarantee

The wiring of our graph is useful in the case where the model fails. To highlight this, we artificially define `n_stuck` below, which replaces the compute node with one that always returns 42, so `verified` can never become true.

The run still terminates. `compute` runs four times — the first pass plus three repairs — and then `route` sees `attempts == MAX_ATTEMPTS` and returns `END`. The final state reports `verified=False`.

That bound comes from `route` counting in Python, not from an instruction the model was asked to follow. While a little heavy-handed, we can see how we can trace out a success condition depending on how the model exits the graph.

In [ ]:
def n_stuck(s):
    return {"value": 42.0, "solves": s["solves"] + 1}

bad, bpath = run_graph(build_graph(n_stuck),
                       {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(bpath))
print(f"verified={bad['verified']}  attempts={bad['attempts']}  solves={bad['solves']}")
assert bad["attempts"] == MAX_ATTEMPTS and not bad["verified"]
print("\nBounded, verified-or-honest, and reproducible. No prompt could have promised that.")

## 2.4 When to reach for a graph

Reach for a graph when the order of operations is a requirement rather than a preference:

| Requirement | What enforces it |
|---|---|
| Every result is verified before it is reported | the edge from `compute` to `verify` |
| At most *N* retries | the router, in Python |
| A failed check loops back rather than giving up | a conditional edge |
| You can see which nodes ran, in order | the streamed path |

The cost is that you now maintain a graph. When the sequence genuinely depends on what the model finds — adaptive meshing, where you cannot say in advance how many refinements are needed — that machinery buys you nothing, and ReAct is the better fit.

## 2.5 Further reading

- **[LangGraph documentation](https://langchain-ai.github.io/langgraph/)** — start here.
- **[Low-level concepts](https://langchain-ai.github.io/langgraph/concepts/low_level/)** — state, nodes, edges and reducers in detail.
- **[Graph API how-to](https://langchain-ai.github.io/langgraph/how-tos/graph-api/)** — the calls used in this notebook.
- **[Persistence and checkpointing](https://langchain-ai.github.io/langgraph/concepts/persistence/)** — how to pause, resume and replay a run. Not used here, but it is the reason graphs get chosen for long jobs.